In [1]:
import pandas as pd
import numpy as np

## File 1: 2000-2010 Intercensal Estimates
Source: https://www.census.gov/data/datasets/time-series/demo/popest/intercensal-2000-2010-counties.html

In [2]:
pop00 = pd.read_csv('../data/raw/co-est00int-tot.csv', encoding='latin-1')

# Keep only counties (SUMLEV=50), not state totals (SUMLEV=40)
pop00 = pop00[pop00['SUMLEV'] == 50].copy()

# Build 5-digit county FIPS
pop00['county_fips'] = pop00['STATE'].astype(str).str.zfill(2) + pop00['COUNTY'].astype(str).str.zfill(3)

# Melt POPESTIMATE columns to long format, keep 2000-2009 only
pop_cols_00 = [c for c in pop00.columns if c.startswith('POPESTIMATE')]
pop00_long = pop00.melt(
    id_vars=['county_fips'], 
    value_vars=pop_cols_00, 
    var_name='attr', 
    value_name='population'
)
pop00_long['year'] = pop00_long['attr'].str.extract(r'(\d{4})').astype(int)
pop00_long = pop00_long[pop00_long['year'] <= 2009][['county_fips', 'year', 'population']]

print(f"2000-2009: Counties={pop00_long['county_fips'].nunique()}, Years={sorted(pop00_long['year'].unique())}")

2000-2009: Counties=3143, Years=[np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009)]


## File 2: 2010-2019 Estimates
Source: https://www.census.gov/data/tables/time-series/demo/popest/2010s-counties-total.html

In [3]:
pop10 = pd.read_csv('../data/raw/co-est2019-alldata.csv', encoding='latin-1')

pop10 = pop10[pop10['SUMLEV'] == 50].copy()
pop10['county_fips'] = pop10['STATE'].astype(str).str.zfill(2) + pop10['COUNTY'].astype(str).str.zfill(3)

pop_cols_10 = [c for c in pop10.columns if c.startswith('POPESTIMATE')]
pop10_long = pop10.melt(
    id_vars=['county_fips'], 
    value_vars=pop_cols_10, 
    var_name='attr', 
    value_name='population'
)
pop10_long['year'] = pop10_long['attr'].str.extract(r'(\d{4})').astype(int)
pop10_long = pop10_long[['county_fips', 'year', 'population']]

print(f"2010-2019: Counties={pop10_long['county_fips'].nunique()}, Years={sorted(pop10_long['year'].unique())}")

2010-2019: Counties=3142, Years=[np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]


## File 3: 2020-2023 from USDA ERS
Source: https://www.ers.usda.gov/data-products/county-level-data-sets/county-level-data-sets-download-data

In [4]:
pop20 = pd.read_csv('../data/raw/PopulationEstimates.csv', encoding='latin-1')

# Filter to population estimate rows only
pop20 = pop20[pop20['Attribute'].str.startswith('POP_ESTIMATE')].copy()
pop20['year'] = pop20['Attribute'].str.extract(r'(\d{4})').astype(int)
pop20['county_fips'] = pop20['FIPStxt'].astype(str).str.zfill(5)

# Remove state-level totals (county portion = 000)
pop20 = pop20[pop20['county_fips'].str.len() == 5]
pop20 = pop20[pop20['county_fips'].str[-3:] != '000']
pop20['population'] = pd.to_numeric(pop20['Value'], errors='coerce')
pop20_long = pop20[['county_fips', 'year', 'population']]

print(f"2020-2023: Counties={pop20_long['county_fips'].nunique()}, Years={sorted(pop20_long['year'].unique())}")

2020-2023: Counties=3222, Years=[np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


## Combine and Compute Growth

In [5]:
pop_all = pd.concat([pop00_long, pop10_long, pop20_long], ignore_index=True)

# Sort and deduplicate (overlap years keep the later vintage)
pop_all = pop_all.sort_values(['county_fips', 'year']).drop_duplicates(
    subset=['county_fips', 'year'], keep='last'
)

# Compute year-over-year population growth
pop_all['pop_growth'] = pop_all.groupby('county_fips')['population'].pct_change()

print(f"COMBINED: Counties={pop_all['county_fips'].nunique()}, "
      f"Years={pop_all['year'].min()}-{pop_all['year'].max()}, "
      f"Rows={len(pop_all)}")
print(f"\npop_growth summary:")
print(pop_all['pop_growth'].describe().round(4))

COMBINED: Counties=3234, Years=2000-2023, Rows=75738

pop_growth summary:
count    72504.0000
mean         0.0024
std          0.0176
min         -0.7677
25%         -0.0052
50%          0.0015
75%          0.0094
max          0.4256
Name: pop_growth, dtype: float64


In [6]:
# Sanity check: Autauga County, AL
pop_all[pop_all['county_fips'] == '01001']

,county_fips,year,population,pop_growth
0,01001,2000,44021.0,NaN
3143,01001,2001,44889.0,0.019718
6286,01001,2002,45909.0,0.022723
9429,01001,2003,46800.0,0.019408
12572,01001,2004,48366.0,0.033462
15715,01001,2005,49676.0,0.027085
18858,01001,2006,51328.0,0.033255
22001,01001,2007,52405.0,0.020983
25144,01001,2008,53277.0,0.016640
28287,01001,2009,54135.0,0.016105


In [7]:
# Save
pop_all.to_csv('../data/processed/county_population_2000_2023.csv', index=False)
print('Saved!')

Saved!
